# Defect Log — train V2 on a GPU

Runs the recipe in `train_v2.py` on Colab's GPU. Nothing here deploys anything:
it produces weights and a predictions file, and the decision comes afterwards
from `docs/EVALUATION.md`.

**Before you start**, on your own machine:

```bash
cd training
./tools/dl.py validate            # must be clean
./tools/dl.py build --commercial --out /tmp/v2set
cd /tmp && zip -qr v2set.zip v2set
```

`build` refuses on any validation error, so a leaking dataset cannot reach
this notebook. Upload `v2set.zip` when the next cell asks.


## 1. Runtime

Runtime → Change runtime type → **T4 GPU**. Then:


In [ ]:
!nvidia-smi -L
!pip -q install ultralytics
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| cuda', torch.cuda.is_available())


## 2. The dataset

Upload the zip built above.


In [ ]:
from google.colab import files
import os, zipfile
up = files.upload()                      # choose v2set.zip
name = next(iter(up))
with zipfile.ZipFile(name) as z:
    z.extractall('/content')
DATA = '/content/v2set/dataset.yaml'
print(open(DATA).read())


### Check what arrived

The class order is the one thing that must not have drifted: the app decodes
channel 4 as manhole and channel 5 as pothole.


In [ ]:
import glob
for split in ('train', 'val', 'test'):
    imgs = glob.glob(f'/content/v2set/images/{split}/*')
    lbls = glob.glob(f'/content/v2set/labels/{split}/*.txt')
    empty = sum(1 for p in lbls if os.path.getsize(p) == 0)
    print(f'{split:6s} {len(imgs):5d} images  {len(lbls):5d} labels  '
          f'{empty:5d} hard negatives')
assert 'names' in open(DATA).read()


## 3. Train

`imgsz=640` and `rect=False` are the two settings most likely to be wrong by
accident: the app stretches a 2340x1080 frame into a 640 square, so the model
must learn on the same distortion. `flipud=0` because the road is always at the
bottom. `seed=0` because a run nobody can repeat is an anecdote.


In [ ]:
from ultralytics import YOLO

RUN = 'v2-2026-09'
model = YOLO('yolov8n.pt')            # same size as the baseline, on purpose
model.train(data=DATA, project='/content/runs', name=RUN,
            epochs=150, imgsz=640, batch=16, patience=30, rect=False,
            fliplr=0.5, flipud=0.0, degrees=5.0, translate=0.1, scale=0.5,
            hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
            mosaic=1.0, close_mosaic=15, seed=0)


## 4. Predictions on the gold test set

Collected down to conf 0.05, not 0.65: the scorer applies the bar, and a file
already cut at 0.65 cannot answer *would a lower bar have caught it?*


In [ ]:
import json, datetime

W = f'/content/runs/{RUN}/weights/best.pt'
best = YOLO(W)
preds = {}
for p in sorted(glob.glob('/content/v2set/images/test/*')):
    stem = os.path.splitext(os.path.basename(p))[0]
    r = best.predict(p, imgsz=640, conf=0.05, verbose=False)[0]
    preds[stem] = [
        {'cls': int(b.cls.item()), 'conf': round(float(b.conf.item()), 6),
         'cx': round(float(b.xywhn[0][0]), 6), 'cy': round(float(b.xywhn[0][1]), 6),
         'w': round(float(b.xywhn[0][2]), 6), 'h': round(float(b.xywhn[0][3]), 6)}
        for b in r.boxes]

doc = {'model': RUN, 'weights': W, 'split': 'test', 'imgsz': 640,
       'collected_from_conf': 0.05,
       'created': datetime.datetime.now(datetime.timezone.utc).isoformat(),
       'predictions': preds}
out = f'/content/preds-{RUN}.json'
json.dump(doc, open(out, 'w'), indent=1)
print(len(preds), 'images,', sum(len(v) for v in preds.values()), 'boxes ->', out)


## 5. Bring it home

Download the weights and the predictions, then score them **locally**, where
the baseline's own predictions are:

```bash
cd training
./tools/dl.py compare preds-baseline.json preds-v2-2026-09.json
```

The rule, stated in advance so it cannot be adjusted afterwards: **a candidate
replaces the baseline only if pothole recall goes up and false positives per
image do not.**


In [ ]:
files.download(out)
files.download(W)
